In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

In [ ]:
CSV_PATH = '/kaggle/input/datasets/orvile/bus-bra-a-breast-ultrasound-dataset/BUSBRA/BUSBRA/bus_data.csv'

us_df = pd.read_csv(CSV_PATH)

print(us_df.shape)
us_df.head()

In [ ]:
us_df.columns = (
    us_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
print(us_df.columns)

In [ ]:
# حذف الصفوف الفارغة من pathology
us_df = us_df.dropna(subset=['pathology'])

# تحويل pathology -> label
us_df['label'] = us_df['pathology'].map({
    'MALIGNANT': 1,
    'BENIGN': 0,
    'malignant': 1,
    'benign': 0
})

# حذف أي قيم غير معروفة
us_df = us_df.dropna(subset=['label'])

# تحويل لـ int
us_df['label'] = us_df['label'].astype(int)

print(us_df['label'].value_counts())


In [ ]:
import os

IMAGE_DIR = '/kaggle/input/datasets/orvile/bus-bra-a-breast-ultrasound-dataset/BUSBRA/BUSBRA/Images'

us_df['image_path'] = us_df['id'].apply(
    lambda x: os.path.join(
        IMAGE_DIR,
        f'{x}.png'
    )
)

In [ ]:
print(us_df[['id', 'image_path']].head())

In [ ]:
print(os.path.exists(us_df['image_path'].iloc[0]))

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample = us_df.sample(4)

plt.figure(figsize=(12,8))

for i, (_, row) in enumerate(sample.iterrows()):

    img = cv2.imread(row['image_path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(2,2,i+1)
    plt.imshow(img)
    plt.title(f"Label: {row['label']}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
us_df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier

import pandas as pd
import numpy as np

# =========================================================
# 📌 FEATURES
# =========================================================

X = us_df[[
    'width',
    'height',
    'hob',
    'k5b',
    'k10b',
    'hop',
    'k5p',
    'k10p'
]]

# Labels
y = us_df['label']

# Image path column
paths = us_df['image_path']

# =========================================================
# 📌 TRAIN / VALIDATION / TEST SPLIT
# =========================================================

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    stratify=y_train_val,
    random_state=42
)

# =========================================================
# 📌 SPLIT COLUMN
# =========================================================

us_df['split'] = 'train'

us_df.loc[X_val.index, 'split'] = 'validation'

us_df.loc[X_test.index, 'split'] = 'test'

# =========================================================
# 📌 MODEL
# =========================================================

cat_base = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_state=42
)

# =========================================================
# 📌 PIPELINE
# =========================================================

cat_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', cat_base)
])

# =========================================================
# 📌 TRAIN
# =========================================================

cat_model.fit(X_train, y_train)

# =========================================================
# 📌 PREDICT ALL DATA
# =========================================================

all_probs = cat_model.predict_proba(X)[:, 1]

# Save PS
us_df['ps_score'] = all_probs

# =========================================================
# 📌 METRICS
# =========================================================

train_probs = cat_model.predict_proba(X_train)[:, 1]
val_probs   = cat_model.predict_proba(X_val)[:, 1]
test_probs  = cat_model.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, train_probs)
val_auc   = roc_auc_score(y_val, val_probs)
test_auc  = roc_auc_score(y_test, test_probs)

print("=" * 60)

print("TRAIN AUC      :", round(train_auc, 4))
print("VALIDATION AUC :", round(val_auc, 4))
print("TEST AUC       :", round(test_auc, 4))

print("=" * 60)

# =========================================================
# 📌 FINAL CSV FILES
# Only: label, path, ps
# =========================================================

train_csv = us_df.loc[X_train.index, [
    'label',
    'image_path',
    'ps_score'
]]

val_csv = us_df.loc[X_val.index, [
    'label',
    'image_path',
    'ps_score'
]]

test_csv = us_df.loc[X_test.index, [
    'label',
    'image_path',
    'ps_score'
]]

# =========================================================
# 📌 SAVE CSV
# =========================================================

train_csv.to_csv(
    "train_predictions.csv",
    index=False
)

val_csv.to_csv(
    "validation_predictions.csv",
    index=False
)

test_csv.to_csv(
    "test_predictions.csv",
    index=False
)

# =========================================================
# 📌 DONE
# =========================================================

print("\n✅ CSV FILES SAVED")

print("train_predictions.csv")
print("validation_predictions.csv")
print("test_predictions.csv")

# =========================================================
# 📌 PREVIEW
# =========================================================

print("\nTrain Preview:")
print(train_csv.head())

print("\nValidation Preview:")
print(val_csv.head())

print("\nTest Preview:")
print(test_csv.head())

In [ ]:
''''
X = us_df[[
    'width',
    'height',
    'hob',
    'k5b',
    'k10b',
    'hop',
    'k5p',
    'k10p'
]]
y = us_df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
us_df['col_split'] = 'train'
us_df.loc[X_test.index, 'col_split'] = 'test' 
''''

In [ ]:
''''
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# =========================
# Logistic Regression Base Model
# =========================
lr_base = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# =========================
# Pipeline
# =========================
lr_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', lr_base)
])

# =========================
# Train
# =========================
lr_model.fit(X_train, y_train)

# =========================
# Predict Probabilities
# =========================
ps_lr = lr_model.predict_proba(X_test)[:, 1]

# =========================
# AUC Score
# =========================
auc_lr = roc_auc_score(y_test, ps_lr)

print("Logistic Regression AUC:", auc_lr)

# =========================
# Add probabilities to dataframe
# =========================
all_ps_lr = lr_model.predict_proba(X)[:, 1]

us_df['PS_ultrasound_LR'] = all_ps_lr

# Preview
print(us_df[['PS_ultrasound_LR']].head())
''''

In [ ]:
''''
RocCurveDisplay.from_predictions(y_test, ps_lr)
plt.title('Ultrasound Logistic Regression ROC')
plt.show()
''''

In [ ]:
''''
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# =========================
# SVM Base Model
# =========================
svm_base = SVC(
    kernel='rbf',
    probability=True,
    random_state=42
)

# =========================
# Pipeline
# =========================
svm_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', svm_base)
])

# =========================
# Train
# =========================
svm_model.fit(X_train, y_train)

# =========================
# Predict Probabilities
# =========================
ps_svm = svm_model.predict_proba(X_test)[:, 1]

# =========================
# AUC Score
# =========================
auc_svm = roc_auc_score(y_test, ps_svm)

print("SVM AUC:", auc_svm)

# =========================
# Add probabilities to dataframe
# =========================
all_ps_svm = svm_model.predict_proba(X)[:, 1]

us_df['PS_ultrasound_SVM'] = all_ps_svm

# Preview
print(us_df[['PS_ultrasound_SVM']].head())
''''

In [ ]:
''''
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

# =========================
# CatBoost Base Model
# =========================
cat_base = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_state=42
)

# =========================
# Pipeline
# =========================
cat_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', cat_base)
])

# =========================
# Train
# =========================
cat_model.fit(X_train, y_train)

# =========================
# Predict Probabilities
# =========================
ps_cat = cat_model.predict_proba(X_test)[:, 1]

# =========================
# AUC Score
# =========================
auc_cat = roc_auc_score(y_test, ps_cat)

print("CatBoost AUC:", auc_cat)

# =========================
# Add probabilities to dataframe
# =========================
all_ps_cat = cat_model.predict_proba(X)[:, 1]

us_df['PS_ultrasound_CAT'] = all_ps_cat

# Preview
print(us_df[['PS_ultrasound_CAT']].head())
''''

In [ ]:
''''
save_cols = []

if 'image_path' in us_df.columns:
    save_cols.append('image_path')

save_cols += [
    'label',
    'PS_ultrasound_CAT',
    'col_split'
]

final_us = us_df[save_cols]
final_us.to_csv('ultrasound_PS_cat.csv', index=False)
print('Saved: ultrasound_PS_cat.csv')
''''